## GenBio-AI

This notebook can be used to format the foundation models which build on genbio-AI's tooling (AIDOCell and scFoundation)

### Environment setup

```bash
uv venv .genbio
source .genbio/bin/activate

uv pip install "modelgenerator==0.1.2" ipykernel "napistu-torch>=0.3.8"
python -m ipykernel install --user --name=genbio --display-name="GenBio-AI (scFoundation/AIDOCell)
```

In [1]:
import os
import logging
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DATA_DIR = "data"
OUTPUT_DIR = "output"
MODEL_PATH = os.path.join(DATA_DIR, "genbioAI")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set environmental variables so models are downloaded to the the model path rather than ~/.cache/huggingface
os.environ['HF_HOME'] = MODEL_PATH                        # Primary control
os.environ['HUGGINGFACE_HUB_CACHE'] = MODEL_PATH          # Hub downloads
os.environ['TRANSFORMERS_CACHE'] = MODEL_PATH             # Transformers-specific

from napistu_torch.load.foundation_model_etl import (
    process_aidocell,
    process_scfoundation,
)
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.constants import (
    AIDOCELL_CLASSES,
    FM_DEFS,
    FOUNDATION_MODEL_NAMES,
)


## AIDOCell

Format each model as a `napistu_torch.load.foundation_model.FoundationModel` instance plust a metadata json and save to disk.

In [2]:
# Process all AIDOCell variants using class names from AIDOCELL_CLASSES
for class_name in [AIDOCELL_CLASSES.THREE_M, AIDOCELL_CLASSES.TEN_M, AIDOCELL_CLASSES.ONE_HUNDRED_M]:
    process_aidocell(class_name, OUTPUT_DIR)

INFO:etl_utils:Extracting: aido_cell_3m
INFO:etl_utils:
1. Loading model and data...
INFO:etl_utils:Loading AIDOCell model
INFO:etl_utils:Loading gene annotations


• standardized 19208/19264 terms


INFO:etl_utils:Formatting model metadata


• standardized 19208/19264 terms


INFO:etl_utils:   19264 genes, 6 layers
INFO:etl_utils:2. Extracting weights...


...

INFO:etl_utils:Formatting model metadata


• standardized 19208/19264 terms


INFO:etl_utils:   19264 genes, 18 layers
INFO:etl_utils:2. Extracting weights...


• standardized 19208/19264 terms


INFO:etl_utils:   Embeddings: (19264, 640)
INFO:etl_utils:   Attention weights: 18 layers × 4 matrices (Q,K,V,O)
INFO:etl_utils:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/AIDOCell_aido_cell_100m_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/AIDOCell_aido_cell_100m_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:etl_utils:Successfully saved all results!


In [3]:
# Create prefix for AIDOCell model (using TEN_M variant as example)
prefix = f"{FOUNDATION_MODEL_NAMES.AIDOCELL}_{AIDOCELL_CLASSES.TEN_M}"

# Load model using FoundationModel.load()
model = FoundationModel.load(OUTPUT_DIR, prefix)

GENES_OF_INTEREST = model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(10000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_attn = model.weights.compute_attention_from_weights(
    layer_idx=5,
    n_heads=model.n_heads,
    vocab_mask=np.array(GENE_MASK)
)

INFO:napistu_torch.load.foundation_models:Loading weights (AIDOCell_aido_cell_10m_weights.npz) and metadata (  AIDOCell_aido_cell_10m_metadata.json) from output_dir (output)
INFO:napistu_torch.load.foundation_models:Successfully loaded all results


## scFoundation

In [4]:
process_scfoundation(output_dir=OUTPUT_DIR, cache_dir=MODEL_PATH)

INFO:etl_utils:Extracting: scFoundation
INFO:etl_utils:
1. Downloading checkpoint from HuggingFace...
INFO:etl_utils:Loading scFoundation checkpoint (gene encoder: gene)
INFO:etl_utils:2. Extracting weights...


• standardized 19208/19264 terms


INFO:etl_utils:Extracting model weights...
INFO:etl_utils:Extracted gene embeddings: (19264, 768)
INFO:etl_utils:Extracted 12 attention layers
INFO:etl_utils:Extracting metadata...
INFO:etl_utils:   19264 genes, 12 layers
INFO:etl_utils:   Embeddings: (19264, 768)
INFO:etl_utils:   Attention weights: 12 layers × 4 matrices (Q,K,V,O)
INFO:etl_utils:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/scFoundation_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scFoundation_metadata.json
INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:etl_utils:Successfully saved all results!


In [5]:
model = FoundationModel.load(OUTPUT_DIR, FOUNDATION_MODEL_NAMES.SCFOUNDATION)

GENES_OF_INTEREST = model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(10000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_attn = model.weights.compute_attention_from_weights(
    layer_idx=5,
    n_heads=model.n_heads,
    vocab_mask=np.array(GENE_MASK)
)

INFO:napistu_torch.load.foundation_models:Loading weights (scFoundation_weights.npz) and metadata (  scFoundation_metadata.json) from output_dir (output)
INFO:napistu_torch.load.foundation_models:Successfully loaded all results
